In [1]:
!pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib markdown2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.9 MB/s eta 0:00:00


In [33]:
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

docs_service = build("docs", "v1")
drive_service = build("drive", "v3")


In [34]:
markdown_text = """
# Product Team Sync - May 15, 2023

## Attendees
- Sarah Chen (Product Lead)
- Mike Johnson (Engineering)
- Anna Smith (Design)
- David Park (QA)

## Agenda

### 1. Sprint Review
* Completed Features
  * User authentication flow
  * Dashboard redesign
  * Performance optimization
    * Reduced load time by 40%
    * Implemented caching solution
* Pending Items
  * Mobile responsive fixes
  * Beta testing feedback integration

### 2. Current Challenges
* Resource constraints in QA team
* Third-party API integration delays
* User feedback on new UI
  * Navigation confusion
  * Color contrast issues

### 3. Next Sprint Planning
* Priority Features
  * Payment gateway integration
  * User profile enhancement
  * Analytics dashboard
* Technical Debt
  * Code refactoring
  * Documentation updates

## Action Items
- [ ] @sarah: Finalize Q3 roadmap by Friday
- [ ] @mike: Schedule technical review for payment integration
- [ ] @anna: Share updated design system documentation
- [ ] @david: Prepare QA resource allocation proposal

## Next Steps
* Schedule individual team reviews
* Update sprint board
* Share meeting summary with stakeholders

## Notes
* Next sync scheduled for May 22, 2023
* Platform demo for stakeholders on May 25
* Remember to update JIRA tickets

---
Meeting recorded by: Sarah Chen
Duration: 45 minutes
"""

In [35]:
import re

def detect_heading(line):
    if line.startswith("# "):
        return 1, line[2:]
    elif line.startswith("## "):
        return 2, line[3:]
    elif line.startswith("### "):
        return 3, line[4:]
    return None, line

def detect_checkbox(line):
    m = re.match(r"- \[( |x)\] (.*)", line)
    if m:
        checked = (m.group(1) == "x")
        return checked, m.group(2)
    return None, line

def detect_bullet(line):
    indent = len(line) - len(line.lstrip(" "))
    stripped = line.strip()
    if stripped.startswith("*") or stripped.startswith("-"):
        return indent // 2, stripped[2:]
    return None, stripped

def detect_footer(line):
    return ("Meeting recorded by" in line) or ("Duration:" in line)


In [36]:
def create_doc(title):
    try:
        doc = docs_service.documents().create(body={"title": title}).execute()
        return doc.get("documentId")
    except Exception as e:
        print("Error creating doc:", e)

doc_id = create_doc("Product Team Sync - Auto Generated")
print("Doc created:", doc_id)


Doc created: 1p1oIva2Wvd5BljCFMld72MVrD_AQSrG_mS63VMwK3h0


In [37]:
# --------------------------
# STEP 6: SIMPLIFIED BULLET HANDLING (NO API BULLET CALLS)
# --------------------------

requests = []
cursor = 1

UNCHECKED = "☐ "
CHECKED = "☑ "

# bullet levels
BULLETS = {
    0: "• ",
    1: "◦ ",
    2: "▪ ",
    3: "▪ ",
}

def add_text(text):
    global cursor
    requests.append({
        "insertText": {
            "location": {"index": cursor},
            "text": text
        }
    })
    cursor += len(text)

lines = markdown_text.split("\n")

for line in lines:
    stripped = line.strip()
    start = cursor

    # empty line
    if stripped == "":
        add_text("\n")
        continue

    # headings
    h_level, text = detect_heading(stripped)
    if h_level:
        add_text(text + "\n")
        requests.append({
            "updateParagraphStyle": {
                "range": {"startIndex": start, "endIndex": cursor},
                "paragraphStyle": {"namedStyleType": f"HEADING_{h_level}"},
                "fields": "namedStyleType"
            }
        })
        continue

    # checkbox (now unicode only)
    cb = detect_checkbox(stripped)
    if isinstance(cb, tuple):
        checked, content = cb
        prefix = CHECKED if checked else UNCHECKED
        add_text(prefix + content + "\n")
        continue

    # bullet detection
    b = detect_bullet(line)
    if isinstance(b, tuple):
        indent, content = b
        bullet_symbol = BULLETS.get(indent, "▪ ")
        add_text(" " * (indent * 2) + bullet_symbol + content + "\n")
        continue

    # footer
    if detect_footer(stripped):
        add_text(stripped + "\n")
        requests.append({
            "updateTextStyle": {
                "range": {"startIndex": start, "endIndex": cursor},
                "textStyle": {
                    "italic": True,
                    "foregroundColor": {"color": {"rgbColor": {"blue": 0.6}}}
                },
                "fields": "italic,foregroundColor"
            }
        })
        continue

    # normal text
    add_text(stripped + "\n")


In [38]:
try:
    docs_service.documents().batchUpdate(
        documentId=doc_id, body={"requests": requests}
    ).execute()

    print("Document successfully created:")
    print(f"https://docs.google.com/document/d/{doc_id}/edit")

except HttpError as e:
    print("Error updating document:", e)


Document successfully created:
https://docs.google.com/document/d/1p1oIva2Wvd5BljCFMld72MVrD_AQSrG_mS63VMwK3h0/edit
